# Classificação com MLPs

Naturalmente, as MLPs também podem ser utilizadas para tarefas de classificação. Para um problema de classificação binária, você só precisa de um único neurônio de saída e da função de ativação logística: a saída será um número entre 0 e 1, que você pode interpretar como a probabilidade estimada da classe positiva. A probabilidade estimada da classe negativa é igual a 1 menos esse número.

As MLPs também podem lidar facilmente com tarefas de classificação binária de múltiplos rótulos. Por exemplo, você pode ter um sistema de classificação de e-mail que prediz se cada e-mail recebido é um "não spam" ou um "spam" e, simultaneamente, se é um e-mail "urgente" ou "não urgente". Nesse caso, você só precisaria de dois neurônios de saída, ambos empregando a função de ativação logística: o primeiro geraria a probabilidade de o e-mail ser spam e o segundo geraria a probabilidade de ser urgente. **Em termos gerais, você dedica um neurônio de saída para cada classe positiva.** Nesse caso, as probabilidades de saída não necessariamente somam 1. Isso permite que o modelo gere a saída de qualquer combinação de rótulos: você pode ter um não spam não urgente; um não spam urgente; um spam não urgente e até um spam urgente.

Se cada instância pode pertencer somente a uma única classe, dentre três ou mais classes possíveis (por exemplo, classes de 0 a 9 para a classificação de imagem com algarismos), é necessário ter um neurônio de saída por classe e usar a função de ativação softmax em toda a camada de saída. A função softmax garantirá que todas as probabilidades estimadas fiquem entre 0 e 1 e que somem até 1. É um classificador multiclasse.

Com relação a função de perda, uma vez que estamos predizendo distribuições de probabilidade, a perda de entropia cruzada (log loss) geralmente é uma boa escolha.

A tabela abaixo, resume a típica arquitetura de uma MLP de classificação:

<p align=center>
<img src="https://github.com/vitorbeltrao/Pictures/blob/main/arq_tipica_mlp_classificacao.png?raw=true" width="50%"></p>

Vamos implementar uma MLP em um exemplo prático. Vamos utilizar a biblioteca [TensorFlow](https://www.tensorflow.org/) para isso.

In [1]:
# importar os pacotes necessários
import pandas as pd
import numpy as np
import tensorflow as tf
import re
import string
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\4YouSee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\4YouSee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\4YouSee\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# carregar o conjunto de dados: IMDB dataset
splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'test': 'plain_text/test-00000-of-00001.parquet', 'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"], engine="fastparquet")
df_test = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["test"], engine="fastparquet")

c:\Users\4YouSee\Desktop\personal work\dl_book\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# verificar o tamanho dos dados de treino e teste
print("Tamanho do conjunto de dados de treino:", len(df_train))
print("Tamanho do conjunto de dados de teste:", len(df_test))

Tamanho do conjunto de dados de treino: 25000
Tamanho do conjunto de dados de teste: 25000


Repare que o conjunto de dados já está dividido em um conjunto de treinamento e em um conjunto de testes, no entanto, não existe um conjunto de validação, vamos criar um.

In [4]:
# criando conjunto de validação
df_train, df_val = train_test_split(
    df_train, test_size=0.20, random_state=42)

In [5]:
# verificar o tamanho dos dados de treino e validação
print("Tamanho do conjunto de dados de treino:", len(df_train))
print("Tamanho do conjunto de dados de validação:", len(df_val))

Tamanho do conjunto de dados de treino: 20000
Tamanho do conjunto de dados de validação: 5000


In [6]:
df_train

,text,label
23311,I borrowed this movie despite its extremely lo...,1
23623,After the unexpected accident that killed an i...,1
1020,On the 1998 summer blockbuster hit BASEketball...,0
12645,Can Scarcely Imagine a Better Movie Than This<...,1
1533,A still famous but decadent actor (Morgan Free...,0
...,...,...
21575,My discovery of the cinema of Jan Svankmajer o...,1
5390,The story is similar to ET: an extraterrestria...,0
860,I have read the novel Reaper of Ben Mezrich a ...,0
15795,Went to see this finnish film and I've got to ...,1


Antes de começarmos a construir a rede neural para treinar nossos dados, vamos fazer uma etapa de pré-processamento necessária!

## Pré-processamento dos dados

In [7]:
class CleanText(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned = []

        for text in X:
            text = str(text).lower()
            text = re.sub(r'\[.*?\]', '', text)
            text = re.sub(r'https?://\S+|www\.\S+', '', text)
            text = re.sub(r'<.*?>+', '', text)
            text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
            text = re.sub(r'\n', '', text)
            text = re.sub(r'\w*\d\w*', '', text)
            text = re.sub(r'\s+', ' ', text).strip()

            cleaned.append(text)

        return cleaned
    

class TokenizeText(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        tokens = []

        for i in range(len(X)):
            token = nltk.word_tokenize(X[i])
            tokens.append(token)

        return tokens
    

class CleanStopwords(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        stop_words = set(stopwords.words("english"))
        cleaned_sentences = []

        for sentence in X:

            cleaned_tokens = []

            for word in sentence:
                if word not in stop_words:
                    cleaned_tokens.append(word)

            cleaned_sentences.append(cleaned_tokens)

        return cleaned_sentences
    

class LemmaWords(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        lemmatizer = WordNetLemmatizer()
        lemma_sentences = []

        for sentence in X:

            lemma_tokens = []

            for word in sentence:
                lemmas = lemmatizer.lemmatize(word, pos ='v')
                lemma_tokens.append(lemmas)

            lemma_sentences.append(lemma_tokens)

        return lemma_sentences


class JoinTokens(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        return [
            " ".join(sentence)
            for sentence in X
        ]

In [8]:
pipeline = Pipeline([
    ("clean", CleanText()),
    ("tokenize", TokenizeText()),
    ("stopwords", CleanStopwords()),
    ("lemmatize", LemmaWords()),
    ("join", JoinTokens()),
    ("tfidf", TfidfVectorizer(
        max_features=30000
    ))
])

In [9]:
X_train = pipeline.fit_transform(df_train["text"].values)
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1834827 stored elements and shape (20000, 30000)>